In [ ]:
from sagemaker.jumpstart.estimator import JumpStartEstimator
from sagemaker.jumpstart.model import JumpStartModel
from sagemaker import Session
from sagemaker import instance_types, hyperparameters

Start session

In [ ]:
session = Session()

model_id, model_version = "meta-textgeneration-llama-3-2-1b-instruct", "*"

In [ ]:
## Update to set the location of your train and validation datasets on S3
train_data_location = '"s3://***/train/'
valid_data_location = '"s3://***/valid/'

In [ ]:
## Find default instance types
instance_type = instance_types.retrieve_default(
    model_id=model_id,
    model_version=model_version,
    scope="training") # ## Find default instance types
print(f'instance type: {instance_type}')

hparams = hyperparameters.retrieve_default(model_id=model_id, model_version=model_version)
print(f'hyperparameters: {hparams})

Set up fine tuning instance

In [ ]:
# pass in None to use the default training_instance_type
training_instance_type = None

# attach a custom session to the Estimator class below
sagemaker_session = session

# accept the eula by passing in Environment values to the Estimator class below
environment = {"accept_eula": "true"}  # set "accept_eula": "true" to accept the EULA for gated models

# optionally, disable output compression by passing to the Estimator class below
disable_output_compression = True

# By default, instruction tuning is set to false. Thus, to use instruction tuning dataset you use:
hyperparameters = {
    "instruction_tuned": "True",
    "epoch": "5",
    "max_input_length": "1024",
    "chat_dataset": "False",
}

# Other parameters can be adjusted e.g. epochs. Increasing max_input_length requires additional 
# training and deployment computational resources - see the llama related link in the readme file.

Define and run estimator

In [ ]:
estimator = JumpStartEstimator(model_id=model_id, 
                               model_version=model_version,
                               hyperparameters=hyperparameters, 
                               instance_type=training_instance_type,
                               environment={"accept_eula": "true"},
                               disable_output_compression=True )

estimator.fit({"training": train_data_location, "validation": valid_data_location})

# The Sagemaker role will save the model artifact in S3. Navigate in the AWS UI to look for your shiny new model. 
# It may be in a bucket with a name like "sagemaker-YourAWSRegion-YourAWSAccountNumber"

When the fitting process completes, the Sagemaker role will save the model artifact in S3. Navigate in the AWS UI to look for your shiny new model. It may be in a bucket with a name like "sagemaker-YourAWSRegion-YourAWSAccountNumber"